In [2]:
from stark_qa import load_skb

DATASET_NAME = "amazon"
skb = load_skb(DATASET_NAME, download_processed=True)

/home/wagnerr/.venvstark/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Loading from /home/wagnerr/.cache/huggingface/hub/datasets--snap-stanford--stark/snapshots/88269e23e90587f99476c5dd74e235a0877e69be/skb/amazon/processed!
Loading cached graph with meta link types ['brand', 'category', 'color']


In [ ]:
from typing import Literal

from pydantic import BaseModel
import random
import csv

import random
from ollama import generate

from neo4j import GraphDatabase


class QueryAmbiguityCheck(BaseModel):
    decision: Literal["KEEP", "DISCARD"]


NUM_QUERIES = 10
OLLAMA_LLM = "gemma4:26b"
QUERY_GEN_PROMPT = """You are an intelligent assistant that generates queries about Amazon items.
I will provide you with a textual document associated with a product entity in a Amazon product recommendation knowledge graph.
Your task is to create a natural-sounding customer query that leads to this entity as the answer.
Try to use many different aspects of the document in your query, such as features, reviews or specifications.
Do not use the product name directly in your query.

Document:
{doc}

Query: """
NEO4J_URI = "bolt://localhost:17687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "X"
SIMILAR_NODES_CYPHER = """MATCH (p1:product {{id: '{id}'}})
MATCH (p2:product)
  SEARCH p2 IN (
    VECTOR INDEX product_index
    FOR p1.embedding
    LIMIT 5
  )
WHERE p1.id <> p2.id
RETURN p2.document AS document"""
QUERY_AMBIGUITY_PROMPT = """You are an intelligent assistant that checks if a query is ambiguous.
I will provide you with a list of documents, each describing a different product.
Document #0 is the document the query was generated with.
The other documents are semantically similar documents.

Please decide:
- "KEEP" if the query is specific enough to only lead to document #0 as the answer.
- "DISCARD" if the query could be understood to lead to one of the other documents as well. 

Query: {query}

Documents:
{documents}"""
OUTPUT_FILE = "../qa_datasets/text_amazon.csv"


driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

product_ids = skb.get_node_ids_by_type("product")
random_product_ids = random.sample(product_ids, k=NUM_QUERIES)

with open(OUTPUT_FILE, "w", encoding="utf-8") as outfile:
    writer = csv.DictWriter(outfile, fieldnames=["id", "query", "answer_ids"])
    writer.writeheader()

    for i, pid in enumerate(random_product_ids):
        doc = skb.get_doc_info(pid, add_rel=False)
        query_gen_prompt = QUERY_GEN_PROMPT.format(doc=doc)
        response = generate(
            model=OLLAMA_LLM,
            prompt=query_gen_prompt,
            options={"temperature": 0.0, "seed": 7},
            think="high",
        )
        generated_query = response.response

        cypher_query = SIMILAR_NODES_CYPHER.format(id=pid)
        records, _, _ = driver.execute_query(
            cypher_query,
        )
        docs_string = f"-----Doc #0-----\n{doc}\n"
        for j, r in enumerate(records):
            docs_string += f"-----Doc #{j+1}-----\n"
            docs_string += r["document"] + "\n"

        query_ambiguity_prompt = QUERY_AMBIGUITY_PROMPT.format(
            query=generated_query, documents=docs_string
        )
        print(query_ambiguity_prompt)
        response = generate(
            model=OLLAMA_LLM,
            prompt=query_ambiguity_prompt,
            options={"temperature": 0.0, "seed": 7},
            think="high",
            format=QueryAmbiguityCheck.model_json_schema(),
        )
        ambiguity_check = QueryAmbiguityCheck.model_validate_json(response.response)
        print(ambiguity_check)

        if ambiguity_check.decision == "KEEP":
            writer.writerow(
                {
                    "id": i,
                    "query": generated_query,
                    "answer_ids": [pid],
                }
            )

You are an intelligent assistant that checks if a query is ambiguous.
I will provide you with a list of documents, each describing a different product.
Document #0 is the document the query was generated with.
The other documents are semantically similar documents.

Please decide:
- "KEEP" if the query is specific enough to only lead to document #0 as the answer.
- "DISCARD" if the query could be understood to lead to one of the other documents as well. 

Query: I'm looking for some breathable golf pants that use moisture-wicking fabric to keep me dry and comfortable during a round. I need something with a polyester and spandex blend for better mobility, and I'd prefer something easy to machine wash that is known for having a great fit.

Documents:
-----Doc #0-----
- product: NIKE Men's Flat Front Pant
- description: Wake up early this Saturday and hit the links in the Nike® Flat Front Pant. Dri-FIT fabric wicks moisture away from the body to keep you calm, collected, and comfortable d